# Taylor–Green 와류 DNS — 한 번에 끝내는 노트북

**사용법:** 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 → **런타임 → 모두 실행(Ctrl+F9)**. 끝입니다.

- 셀은 3개입니다. ① 정의(몇 초) ② 계산(**약 80분**, 이미 끝난 계산은 `results.pkl`에서 불러와 건너뜀) ③ 분석·그림·요약·압축(1분 이내).
- ②가 오래 걸려서 중간에 끊기면, 다시 **모두 실행**만 하면 이어서 합니다(저장된 것은 건너뜀). 단 코랩 런타임이 완전히 초기화되면 `results.pkl`도 사라집니다.
- 시간을 줄이려면 ②의 맨 위 `RUN_DT_256 = False`로 바꾸세요(약 26분 절약). 320³를 빼려면 `RUN_LIST`에서 `(320, 1600)`을 지우세요(약 33분 절약).
- 끝나면 ③이 `tg_outputs.zip`(그림 7장 + summary.txt + results.pkl)을 자동 다운로드합니다. **summary.txt의 내용을 그대로 보내 주시면** 논문 수치를 채웁니다.


In [ ]:
# ===== 셀 1: 정의 (몇 초) =====
import time, pickle, os, json, zipfile, io
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

try:
    import cupy as cp
    xp = cp; GPU = True
    print("GPU(cupy) 사용")
except Exception:
    xp = np; GPU = False
    print("CPU(numpy) 사용 -- 64^3 이하만 현실적입니다")

def to_np(a):
    return cp.asnumpy(a) if GPU else np.asarray(a)

def free_gpu():
    if GPU:
        cp.get_default_memory_pool().free_all_blocks()

def ifrk4_step(uh, dt, L, nonlin):
    """적분 인자 RK4: 점성항은 정확히, 비선형항만 RK4."""
    E = xp.exp(L * dt / 2); E2 = E * E
    a = dt * nonlin(uh)
    b = dt * nonlin(E * (uh + a / 2))
    c = dt * nonlin(E * uh + b / 2)
    d = dt * nonlin(E2 * uh + E * c)
    return E2 * uh + (E2 * a + 2 * E * (b + c) + d) / 6

class Spectral3D:
    """주기 상자 [0,2pi]^3, du/dt = P[u x w] + nu*lap(u), 2/3 비에일리어싱(|k_i| < N/3)."""
    def __init__(self, N, nu):
        self.N, self.nu = N, nu
        k = xp.fft.fftfreq(N, 1.0 / N); kz = xp.fft.rfftfreq(N, 1.0 / N)
        self.kx, self.ky, self.kz = xp.meshgrid(k, k, kz, indexing="ij")
        self.k2 = self.kx**2 + self.ky**2 + self.kz**2
        self.k2i = xp.where(self.k2 > 0, 1.0 / xp.where(self.k2 > 0, self.k2, 1.0), 0.0)
        kmax = N / 3.0
        self.mask = ((xp.abs(self.kx) < kmax) & (xp.abs(self.ky) < kmax) & (self.kz < kmax)).astype(float)
        self.L = -nu * self.k2
        w = xp.full(self.k2.shape, 2.0); w[..., 0] = 1.0
        if N % 2 == 0: w[..., -1] = 1.0
        self.w = w
        self.shell = np.rint(np.sqrt(to_np(self.k2))).astype(int).ravel()
        self.nshell = N // 2 + 1
    def phys(self, fh):
        return xp.fft.irfftn(fh, s=(self.N,) * 3, axes=(1, 2, 3))
    def vorticity_hat(self, uh):
        kx, ky, kz = self.kx, self.ky, self.kz
        return 1j * xp.stack([ky * uh[2] - kz * uh[1], kz * uh[0] - kx * uh[2], kx * uh[1] - ky * uh[0]])
    def nonlin(self, uh):
        u = self.phys(uh); w = self.phys(self.vorticity_hat(uh))
        cr = xp.stack([u[1] * w[2] - u[2] * w[1], u[2] * w[0] - u[0] * w[2], u[0] * w[1] - u[1] * w[0]])
        nh = xp.fft.rfftn(cr, axes=(1, 2, 3)) * self.mask
        kdn = self.kx * nh[0] + self.ky * nh[1] + self.kz * nh[2]
        return nh - xp.stack([self.kx, self.ky, self.kz]) * (kdn * self.k2i)
    def taylor_green(self):
        N = self.N; x = xp.arange(N) * (2 * np.pi / N)
        X, Y, Z = xp.meshgrid(x, x, x, indexing="ij")
        u = xp.stack([xp.sin(X) * xp.cos(Y) * xp.cos(Z), -xp.cos(X) * xp.sin(Y) * xp.cos(Z), xp.zeros_like(X)])
        return xp.fft.rfftn(u, axes=(1, 2, 3)) * self.mask
    def diagnostics(self, uh):
        N6 = float(self.N) ** 6
        mode = (xp.abs(uh) ** 2).sum(axis=0) * self.w
        E = float(0.5 * mode.sum() / N6); Om = float(0.5 * (self.k2 * mode).sum() / N6)
        om = self.phys(self.vorticity_hat(uh)); u = self.phys(uh)
        wmax = float(xp.sqrt((om**2).sum(axis=0)).max()); umax = float(xp.sqrt((u**2).sum(axis=0)).max())
        return E, 2 * self.nu * Om, wmax, Om, umax
    def spectrum(self, uh):
        N6 = float(self.N) ** 6
        mode = 0.5 * (xp.abs(uh) ** 2).sum(axis=0) * self.w / N6
        Ek = np.bincount(self.shell, weights=to_np(mode).ravel(), minlength=self.nshell + 2)
        return Ek[: self.nshell]

def run_3d(N, Re, dtmax=0.01, T=10.0, t_rec=0.1, cfl=0.5, spec_times=(0, 2, 4, 6, 8, 8.5, 9, 9.5, 10), verbose=True):
    nu = 1.0 / Re; S = Spectral3D(N, nu); uh = S.taylor_green(); dx = 2 * np.pi / N
    t = 0.0; rec = []; spectra = {}
    E, eps, wmax, Om, umax = S.diagnostics(uh); rec.append((t, E, eps, wmax, Om, umax)); spectra[0.0] = S.spectrum(uh)
    nxt = t_rec; left = [s for s in spec_times if s > 0]
    dt = min(dtmax, cfl * dx / umax); t0 = time.time(); nstep = 0; nprint = 0
    while t < T - 1e-12:
        h = min(dt, nxt - t)
        uh = ifrk4_step(uh, h, S.L, S.nonlin); t += h; nstep += 1
        if t >= nxt - 1e-12:
            E, eps, wmax, Om, umax = S.diagnostics(uh)
            if not np.isfinite(E):
                print(f"  !! 발산 N={N} Re={Re} t={t:.2f}"); break
            rec.append((t, E, eps, wmax, Om, umax))
            dt = min(dtmax, cfl * dx / max(umax, 1e-6)); nxt += t_rec
            if left and t >= left[0] - 1e-9:
                spectra[round(left[0], 3)] = S.spectrum(uh); left.pop(0)
            if verbose and int(t) > nprint and int(t) % 2 == 0:
                nprint = int(t); print(f"    t={t:5.2f}  eps={eps:.5f}  wmax={wmax:7.2f}  ({time.time()-t0:.0f}s)")
    a = np.array(rec)
    out = {"N": N, "Re": Re, "dtmax": dtmax, "t": a[:, 0], "E": a[:, 1], "eps": a[:, 2], "wmax": a[:, 3],
           "Om": a[:, 4], "umax": a[:, 5], "spectra": spectra, "walltime": time.time() - t0, "nstep": nstep}
    free_gpu()
    return out

class Spectral2D:
    def __init__(self, N, nu):
        self.N, self.nu = N, nu
        k = xp.fft.fftfreq(N, 1.0 / N); kz = xp.fft.rfftfreq(N, 1.0 / N)
        self.kx, self.ky = xp.meshgrid(k, kz, indexing="ij")
        self.k2 = self.kx**2 + self.ky**2
        self.k2i = xp.where(self.k2 > 0, 1.0 / xp.where(self.k2 > 0, self.k2, 1.0), 0.0)
        kmax = N / 3.0
        self.mask = ((xp.abs(self.kx) < kmax) & (self.ky < kmax)).astype(float)
        self.L = -nu * self.k2
        w = xp.full(self.k2.shape, 2.0); w[:, 0] = 1.0
        if N % 2 == 0: w[:, -1] = 1.0
        self.w = w
    def nonlin(self, wh):
        psih = wh * self.k2i; s = (self.N,) * 2
        u = xp.fft.irfft2(1j * self.ky * psih, s=s); v = xp.fft.irfft2(-1j * self.kx * psih, s=s)
        wx = xp.fft.irfft2(1j * self.kx * wh, s=s); wy = xp.fft.irfft2(1j * self.ky * wh, s=s)
        return -xp.fft.rfft2(u * wx + v * wy) * self.mask
    def enstrophy(self, wh):
        return float(0.5 * (self.w * xp.abs(wh) ** 2).sum() / self.N**4)
    def energy(self, wh):
        return float(0.5 * (self.w * self.k2i * xp.abs(wh) ** 2).sum() / self.N**4)

def run_2d(N, Re, E0=0.125, T=10.0, t_rec=0.1, cfl=0.5, seed=0):
    S = Spectral2D(N, 1.0 / Re); rng = np.random.default_rng(seed)
    kx = to_np(S.kx); ky = to_np(S.ky); mag = np.sqrt(kx**2 + ky**2)
    ph = np.exp(2j * np.pi * rng.random(kx.shape))
    wh = xp.asarray(((mag > 0) & (mag <= 4)) * ph) * S.mask
    wh = xp.fft.rfft2(xp.fft.irfft2(wh, s=(N, N))) * S.mask
    wh = wh * np.sqrt(E0 / S.energy(wh))
    t = 0.0; out = [(0.0, S.enstrophy(wh))]; nxt = t_rec; dx = 2 * np.pi / N; dt = 0.01
    while t < T - 1e-12:
        h = min(dt, nxt - t); wh = ifrk4_step(wh, h, S.L, S.nonlin); t += h
        if t >= nxt - 1e-12:
            out.append((t, S.enstrophy(wh))); psih = wh * S.k2i; s = (N, N)
            um = float(xp.sqrt(xp.fft.irfft2(1j * S.ky * psih, s=s) ** 2 + xp.fft.irfft2(-1j * S.kx * psih, s=s) ** 2).max())
            dt = min(0.01, cfl * dx / max(um, 1e-6)); nxt += t_rec
    return np.array(out)

def at(r, key, t_target):
    i = int(np.argmin(np.abs(r["t"] - t_target))); return r["t"][i], r[key][i]

def trapz(y, x):
    return float(np.sum(0.5 * (y[1:] + y[:-1]) * np.diff(x)))

print("정의 완료")

In [ ]:
# ===== 셀 2: 계산 (약 80분, 저장된 것은 건너뜀) =====
T_END = 10.0
RUN_LIST = [(64, 100), (64, 200), (64, 400), (128, 400), (128, 800), (192, 800), (256, 1600), (320, 1600)]            # (격자 수 N, Re)
DT_SMALL = (64, 400, [0.005, 0.0025])            # (N, Re, [dt 상한들]) : 시간 간격 수렴 시험 (몇 분)
RUN_DT_256 = True        # 256^3 Re=1600 에서 dt=0.005 재계산 (약 26분). 시간이 없으면 False
DT_BIG = (256, 1600, 0.005)

results = pickle.load(open("results.pkl", "rb")) if os.path.exists("results.pkl") else []
dt_tests = pickle.load(open("dt_tests.pkl", "rb")) if os.path.exists("dt_tests.pkl") else {}
print("불러온 결과:", [(r["N"], r["Re"]) for r in results], "| dt 시험:", list(dt_tests))

def have(N, Re, dt=0.01):
    return any(r["N"] == N and r["Re"] == Re and abs(r.get("dtmax", 0.01) - dt) < 1e-12 for r in results)

t_all = time.time()
for N, Re in RUN_LIST:
    if have(N, Re):
        print(f"[건너뜀] N={N}, Re={Re} (이미 있음)"); continue
    print(f"=== N={N}, Re={Re} ===")
    r = run_3d(N, Re, T=T_END)
    print(f"  완료 {r['walltime']/60:.1f}분")
    results.append(r); pickle.dump(results, open("results.pkl", "wb"))

N_, Re_, dts_ = DT_SMALL
for dtm in dts_:
    key = f"{N_}_{Re_}_{dtm}"
    if key in dt_tests:
        print(f"[건너뜀] dt 시험 {key}"); continue
    print(f"=== dt 시험: N={N_}, Re={Re_}, dt 상한={dtm} ===")
    dt_tests[key] = run_3d(N_, Re_, dtmax=dtm, T=T_END, verbose=False)
    pickle.dump(dt_tests, open("dt_tests.pkl", "wb"))

if RUN_DT_256:
    N_, Re_, dtm = DT_BIG
    key = f"{N_}_{Re_}_{dtm}"
    if key in dt_tests:
        print(f"[건너뜀] dt 시험 {key}")
    else:
        print(f"=== dt 시험: N={N_}, Re={Re_}, dt 상한={dtm} (약 26분) ===")
        dt_tests[key] = run_3d(N_, Re_, dtmax=dtm, T=T_END)
        pickle.dump(dt_tests, open("dt_tests.pkl", "wb"))

print(f"\n계산 단계 완료. 이번 실행에 쓴 시간: {(time.time()-t_all)/60:.1f}분")

In [ ]:
# ===== 셀 3: 분석 · 그림 · 요약 · 압축 =====
LOG = []
def P(*a):
    s = " ".join(str(x) for x in a); print(s); LOG.append(s)

base_runs = [r for r in results if abs(r.get("dtmax", 0.01) - 0.01) < 1e-12]
def kme(r):
    ip = int(np.argmax(r["eps"])); nu = 1.0 / r["Re"]
    return r["N"] / 3 * (nu**3 / r["eps"][ip]) ** 0.25
def verdict(k):
    return "충분" if k >= 1.1 else ("경계" if k >= 0.8 else "부족")
base_runs.sort(key=lambda r: (r["Re"], r["N"]))

# 신뢰할 수 있는 가장 높은 해상도(Re마다)
best = {}
for r in base_runs:
    if r["Re"] not in best or r["N"] > best[r["Re"]]["N"]:
        best[r["Re"]] = r
Res_sorted = sorted(best)

P("=" * 78); P("A. 해상도 진단 (표 3)"); P("=" * 78)
P(f"{'N':>4} {'Re':>5} {'eps_peak':>9} {'t_peak':>7} {'kmax*eta':>9} 판정")
for r in base_runs:
    ip = int(np.argmax(r["eps"]))
    P(f"{r['N']:>4} {r['Re']:>5.0f} {r['eps'][ip]:>9.5f} {r['t'][ip]:>7.2f} {kme(r):>9.2f} {verdict(kme(r))}")

P("\n" + "=" * 78); P("B. 수치표 (표 5 등): 최대 와도, BKM 적분, ω_rms 비, 에너지 수지"); P("=" * 78)
P(f"{'N':>4} {'Re':>5} {'wmax(t~8)':>10} {'wmax,max':>9} {'BKM(t<=10)':>11} {'wmax/wrms@pk':>13} {'umax,max':>9} {'E0-E(T)':>9} {'int eps':>9} {'rel':>8}")
for r in base_runs:
    nu = 1.0 / r["Re"]; ip = int(np.argmax(r["eps"]))
    wrms = np.sqrt(r["eps"][ip] / nu)
    tend = min(10.0, r["t"][-1]); m = r["t"] <= tend + 1e-9
    bkm = trapz(r["wmax"][m], r["t"][m]); dE = r["E"][0] - r["E"][m][-1]; I = trapz(r["eps"][m], r["t"][m])
    um = f"{r['umax'].max():9.3f}" if "umax" in r else f"{'n/a':>9}"
    P(f"{r['N']:>4} {r['Re']:>5.0f} {at(r,'wmax',8.0)[1]:>10.2f} {r['wmax'].max():>9.2f} {bkm:>11.1f} {r['wmax'][ip]/wrms:>13.2f} {um} {dE:>9.5f} {I:>9.5f} {abs(dE-I)/dE:>8.1e}")

P("\n" + "=" * 78); P("C. 최대 와도·BKM 적분의 Re 지수 (Re ≥ 400, k_max·η ≥ 0.95인 실행만)"); P("=" * 78)
good = {R: r for R, r in best.items() if R >= 400 and kme(r) >= 0.95}
Rs = sorted(good)
if len(Rs) >= 2:
    w8 = np.array([at(good[R], "wmax", 8.0)[1] for R in Rs]); B = np.array([trapz(good[R]["wmax"], good[R]["t"]) for R in Rs])
    pw = np.polyfit(np.log(Rs), np.log(w8), 1)[0]; pb = np.polyfit(np.log(Rs), np.log(B), 1)[0]
    P("Re:", Rs); P("wmax(t~8):", np.round(w8, 2), f" -> Re^{pw:.2f}"); P("BKM:", np.round(B, 1), f" -> Re^{pb:.2f}")
    ratios = [good[R]["wmax"][int(np.argmax(good[R]['eps']))] / np.sqrt(good[R]['eps'].max() * R) for R in Rs]
    P("wmax/wrms @ 소산율 피크 시각:", np.round(ratios, 2))
else:
    P("(해당 실행이 2개 미만이라 생략)")

P("\n" + "=" * 78); P("D. 같은 Re, 다른 해상도 비교 (표 4). 괄호는 높은 해상도 값에 대한 상대차"); P("=" * 78)
for Re in sorted(set(r["Re"] for r in base_runs)):
    grp = sorted([r for r in base_runs if r["Re"] == Re], key=lambda r: r["N"])
    if len(grp) < 2: continue
    lo, hi = grp[-2], grp[-1]
    P(f"Re={Re:.0f}: N={lo['N']} vs N={hi['N']}")
    for t_ in (6, 8, 10):
        a, b = at(lo, "eps", t_)[1], at(hi, "eps", t_)[1]; c, d = at(lo, "wmax", t_)[1], at(hi, "wmax", t_)[1]
        P(f"   t={t_:>2}  eps {a:.5f} / {b:.5f} ({100*abs(a-b)/b:.1f}%)   wmax {c:.2f} / {d:.2f} ({100*abs(c-d)/d:.1f}%)")
    a, b = lo["eps"].max(), hi["eps"].max()
    P(f"   eps 피크 {a:.5f} / {b:.5f} ({100*abs(a-b)/b:.1f}%),  피크 시각 {lo['t'][lo['eps'].argmax()]:.2f} / {hi['t'][hi['eps'].argmax()]:.2f}")

P("\n" + "=" * 78); P("E. 스펙트럼 지수 n: E(k) = A exp(-beta k) k^(-n), 적합 구간 2 ≤ k ≤ 0.8 N/3 (문헌: R>1000에서 n=1.9±0.3)"); P("=" * 78)
def fit_n(r):
    ip = int(np.argmax(r["eps"])); tp = r["t"][ip]
    ts = min(r["spectra"], key=lambda s: abs(s - tp)); Ek = np.asarray(r["spectra"][ts]); k = np.arange(len(Ek))
    m = (k >= 2) & (k <= 0.8 * r["N"] / 3) & (Ek > 0); kk = k[m].astype(float)
    X = np.vstack([np.ones_like(kk), -kk, -np.log(kk)]).T
    coef, *_ = np.linalg.lstsq(X, np.log(Ek[m]), rcond=None)
    return ts, coef[2], coef[1], (kk.min(), kk.max())
for r in base_runs:
    ts, n, beta, rng = fit_n(r)
    P(f"  N={r['N']:>3} Re={r['Re']:>5.0f} t={ts:g}  n={n:.2f}  beta={beta:.3f}  k={rng[0]:.0f}..{rng[1]:.0f}  (k_max·η={kme(r):.2f})")

P("\n" + "=" * 78); P("F. 시간 간격 수렴 (Δt 상한 0.01 기준 대비 최대 상대 편차)"); P("=" * 78)
def compare(a, b, label):
    d_eps = np.max(np.abs(a["eps"] - b["eps"])) / np.max(b["eps"]); d_w = np.max(np.abs(a["wmax"] - b["wmax"])) / np.max(b["wmax"])
    d_pk = abs(a["eps"].max() - b["eps"].max()) / b["eps"].max()
    P(f"  {label}: max|Δε|/ε_max={d_eps:.2e}  ε피크 상대차={d_pk:.2e}  max|Δ‖ω‖∞|/‖ω‖∞,max={d_w:.2e}")
for key, r2 in dt_tests.items():
    N_, Re_, dtm = key.split("_")
    base = [r for r in base_runs if r["N"] == int(N_) and abs(r["Re"] - float(Re_)) < 1e-9]
    if base: compare(r2, base[0], f"N={N_} Re={Re_} dt상한={dtm} vs 0.01")
P("  Courant 수 C = dt*umax/dx (umax≈1):", {N: round(0.01 / (2 * np.pi / N), 2) for N in (64, 128, 192, 256, 320)})

P("\n" + "=" * 78); P("G. 벤치마크용 Re=1600 값 (HiOCFD 기준 파일과 비교할 때 사용)"); P("=" * 78)
for r in base_runs:
    if abs(r["Re"] - 1600) < 1e-9:
        ip = int(np.argmax(r["eps"])); P(f"  N={r['N']}: eps 피크 {r['eps'][ip]:.5f} at t={r['t'][ip]:.2f},  ‖ω‖∞(t≈8) = {at(r,'wmax',8.0)[1]:.2f}")
ref_path = "spectral_Re1600_512.gdiag"
if os.path.exists(ref_path):
    P("  기준 파일 발견:", ref_path); P(open(ref_path).read()[:300])
else:
    P("  (기준 파일 spectral_Re1600_512.gdiag 없음 -- 그림 비교는 생략)")

# ---------------- 그림 (논문 번호와 같은 파일 이름) ----------------
P("\n그림 저장 중 ...")
def label(r): return f"N={r['N']}, Re={r['Re']:.0f}"
maxN = {Re: best[Re]["N"] for Re in best}
plt.figure(figsize=(8, 5))
for r in base_runs:
    plt.plot(r["t"], r["eps"], label=label(r), ls="-" if r["N"] == maxN[r["Re"]] else "--")
plt.xlabel("t"); plt.ylabel(r"$\varepsilon=\nu\langle|\omega|^2\rangle$"); plt.title("Taylor–Green: energy dissipation rate")
plt.legend(fontsize=8); plt.grid(alpha=.3); plt.tight_layout(); plt.savefig("fig1_dissipation.png", dpi=150); plt.close()

peaks = [best[R]["eps"].max() for R in Res_sorted]; tpk = [best[R]["t"][np.argmax(best[R]["eps"])] for R in Res_sorted]
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].semilogx(Res_sorted, peaks, "o-"); ax[0].set_xlabel("Re"); ax[0].set_ylabel(r"max $\varepsilon$"); ax[0].set_title("Peak dissipation rate"); ax[0].grid(alpha=.3)
ax[1].semilogx(Res_sorted, tpk, "s-"); ax[1].set_xlabel("Re"); ax[1].set_ylabel("time of peak"); ax[1].set_title("Time of peak dissipation"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig("fig2_peak_vs_Re.png", dpi=150); plt.close()

pairs = {}
for r in base_runs: pairs.setdefault(r["Re"], []).append(r)
pairs = {k: sorted(v, key=lambda x: x["N"]) for k, v in pairs.items() if len(v) > 1}
if pairs:
    fig, ax = plt.subplots(1, len(pairs), figsize=(6 * len(pairs), 4.2), squeeze=False)
    for a, (Re, v) in zip(ax[0], pairs.items()):
        for r in v: a.plot(r["t"], r["eps"], label=f"N={r['N']}")
        a.set_title(f"Re={Re:.0f}"); a.set_xlabel("t"); a.set_ylabel(r"$\varepsilon$"); a.legend(); a.grid(alpha=.3)
    plt.tight_layout(); plt.savefig("fig3_convergence.png", dpi=150); plt.close()

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for r in base_runs:
    ax[0].semilogy(r["t"], r["wmax"], label=label(r))
    bkm = np.concatenate([[0], np.cumsum(0.5 * (r["wmax"][1:] + r["wmax"][:-1]) * np.diff(r["t"]))]); ax[1].plot(r["t"], bkm, label=label(r))
ax[0].set_title("Max vorticity"); ax[0].set_xlabel("t"); ax[0].set_ylabel(r"$\|\omega\|_\infty$")
ax[1].set_title("BKM integral"); ax[1].set_xlabel("t"); ax[1].set_ylabel(r"$\int_0^t\|\omega\|_\infty dt'$")
for a in ax: a.grid(alpha=.3); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig("fig4_vorticity_bkm.png", dpi=150); plt.close()

plt.figure(figsize=(7.5, 5))
for R in Res_sorted:
    r = best[R]; plt.plot(r["t"], r["Om"] / r["Om"][0], label=f"3D TG, Re={R:.0f}")
o2 = run_2d(256, Res_sorted[-1], T=max(r["t"][-1] for r in base_runs))
plt.plot(o2[:, 0], o2[:, 1] / o2[0, 1], "k--", lw=2, label=f"2D random, Re={Res_sorted[-1]:.0f}")
plt.yscale("log"); plt.xlabel("t"); plt.ylabel(r"$\Omega(t)/\Omega(0)$"); plt.title("Enstrophy: 2D vs 3D"); plt.legend(fontsize=8)
plt.grid(alpha=.3, which="both"); plt.tight_layout(); plt.savefig("fig5_2d_vs_3d.png", dpi=150); plt.close()
P(f"  2D 엔스트로피 Ω(t_end)/Ω(0) = {o2[-1,1]/o2[0,1]:.3f}")

plt.figure(figsize=(7, 5))
for R in Res_sorted:
    r = best[R]; tp = r["t"][np.argmax(r["eps"])]; ts = min(r["spectra"], key=lambda s: abs(s - tp)); Ek = r["spectra"][ts]; k = np.arange(len(Ek))
    plt.loglog(k[1:], Ek[1:], label=f"Re={R:.0f} (N={r['N']}, t={ts:g})")
rr = best[Res_sorted[-1]]; tsr = min(rr["spectra"], key=lambda s: abs(s - rr["t"][np.argmax(rr["eps"])]))
kk = np.array([3, 30.0]); plt.loglog(kk, rr["spectra"][tsr][3] * 3 ** (5 / 3) * 0.5 * kk ** (-5 / 3), "k--", label=r"$k^{-5/3}$ (reference)")
plt.xlabel("k"); plt.ylabel("E(k)"); plt.title("Energy spectrum at peak dissipation"); plt.legend(fontsize=7); plt.grid(alpha=.3, which="both")
plt.tight_layout(); plt.savefig("fig6_spectrum.png", dpi=150); plt.close()

Re_line = np.logspace(2, 8, 100); plt.figure(figsize=(8, 5))
plt.loglog(Re_line, Re_line ** 2.25, "b-", label=r"$N_{total}\sim Re^{9/4}$")
for r in base_runs: plt.loglog(r["Re"], r["N"] ** 3, "ro")
plt.loglog([], [], "ro", label="runs in this notebook")
for Re_a, nm in [(1e5, "small RC/UAV wing (approx.)"), (1e7, "airliner wing (approx.)")]:
    plt.axvline(Re_a, color="gray", ls=":"); plt.text(Re_a * 1.1, 1e3, f"{nm}  Re~{Re_a:.0e}", fontsize=8, rotation=90, va="bottom")
plt.xlabel("Re"); plt.ylabel("required grid points (order of magnitude)"); plt.title("DNS cost estimate"); plt.legend(); plt.grid(alpha=.3, which="both")
plt.tight_layout(); plt.savefig("fig7_cost.png", dpi=150); plt.close()

# ---------------- 요약 저장 + 압축 + 다운로드 ----------------
open("summary.txt", "w", encoding="utf-8").write("\n".join(LOG))
files_out = [f for f in ["fig1_dissipation.png", "fig2_peak_vs_Re.png", "fig3_convergence.png", "fig4_vorticity_bkm.png",
                         "fig5_2d_vs_3d.png", "fig6_spectrum.png", "fig7_cost.png", "summary.txt", "results.pkl", "dt_tests.pkl"] if os.path.exists(f)]
with zipfile.ZipFile("tg_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in files_out: z.write(f)
print("\n저장된 파일:", files_out)
try:
    from google.colab import files
    files.download("tg_outputs.zip")
except Exception as e:
    print("(자동 다운로드는 코랩에서만 동작합니다)", type(e).__name__)
try:
    from IPython.display import Image, display
    for f in files_out:
        if f.endswith(".png"): display(Image(f))
except Exception:
    pass